In [1]:
import torch
from torch import nn, optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture
from collections import deque, defaultdict

# CAE

In [2]:
train = pd.read_csv('../../train_CADE_cod.csv')
val = pd.read_csv('../../val_CADE_cod.csv')
test = pd.read_csv('../../test_CADE_cod.csv')

In [3]:
classes_out = ['Infilteration','DoS attacks-SlowHTTPTest']
train = train.drop(index=train[train['Label'].isin(classes_out)].index)
val = val.drop(index=val[val['Label'].isin(classes_out)].index)
test = test.drop(index=test[test['Label'].isin(classes_out)].index)

In [4]:
train['Label'] = train['Label'].replace('FTP-BruteForce','BruteForce')
train['Label'] = train['Label'].replace('SSH-Bruteforce','BruteForce')
train['Label'] = train['Label'].replace('DoS attacks-GoldenEye','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Slowloris','DoS')
# train['Label'] = train['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Hulk','DoS')
train['Label'] = train['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-HOIC','DDoS')
train['Label'] = train['Label'].replace('Brute Force -Web','Web')
train['Label'] = train['Label'].replace('Brute Force -XSS','Web')
train['Label'] = train['Label'].replace('SQL Injection','Web')


val['Label'] = val['Label'].replace('FTP-BruteForce','BruteForce')
val['Label'] = val['Label'].replace('SSH-Bruteforce','BruteForce')
val['Label'] = val['Label'].replace('DoS attacks-GoldenEye','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Slowloris','DoS')
# val['Label'] = val['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Hulk','DoS')
val['Label'] = val['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-HOIC','DDoS')
val['Label'] = val['Label'].replace('Brute Force -Web','Web')
val['Label'] = val['Label'].replace('Brute Force -XSS','Web')
val['Label'] = val['Label'].replace('SQL Injection','Web')


test['Label'] = test['Label'].replace('FTP-BruteForce','BruteForce')
test['Label'] = test['Label'].replace('SSH-Bruteforce','BruteForce')
test['Label'] = test['Label'].replace('DoS attacks-GoldenEye','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Slowloris','DoS')
# test['Label'] = test['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Hulk','DoS')
test['Label'] = test['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-HOIC','DDoS')
test['Label'] = test['Label'].replace('Brute Force -Web','Web')
test['Label'] = test['Label'].replace('Brute Force -XSS','Web')
test['Label'] = test['Label'].replace('SQL Injection','Web')

In [20]:
len(test['Label'].values)

649947

In [5]:
labels_str = [i for i in train['Label'].value_counts().index]
print(labels_str)

for i, l in enumerate(labels_str):
    train['Label'] = train['Label'].replace(l, i)
    val['Label'] = val['Label'].replace(l, i)
    test['Label'] = test['Label'].replace(l, i)

['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']


C:\Users\linco\AppData\Local\Temp\ipykernel_50728\2307520911.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['Label'] = train['Label'].replace(l, i)
C:\Users\linco\AppData\Local\Temp\ipykernel_50728\2307520911.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['Label'] = val['Label'].replace(l, i)
C:\Users\linco\AppData\Local\Temp\ipykernel_50728\2307520911.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result

In [16]:
test['Label']

0         0
1         0
2         1
3         1
5         0
         ..
710046    1
710047    0
710048    0
710049    3
710050    2
Name: Label, Length: 649947, dtype: int64

# CAE CADE (margin 10 e lambda 0.1)

In [6]:
def pick_pair(embeddings, labels, pick_embeddings, pick_labels):
    """
    Gera todos os pares entre embeddings e pick_embeddings,
    com labels_pair indicando se os pares são positivos (0) ou negativos (1).
    """
    N, D = embeddings.shape
    M = pick_embeddings.shape[0]

    # Expande para todas as combinações possíveis
    A = embeddings.unsqueeze(0).expand(M, N, D).reshape(M * N, D)
    B = pick_embeddings.unsqueeze(1).expand(M, N, D).reshape(M * N, D)

    labels_exp = labels.unsqueeze(0).expand(M, N)
    pick_labels_exp = pick_labels.unsqueeze(1).expand(M, N)
    labels_pair = (labels_exp != pick_labels_exp).reshape(-1).float()

    return A, B, labels_pair

def ContrastiveLoss(input1, input2, y, margin=1.0):
    # y = 0 quando input1 e input2 são da mesma classe e y = 1 caso contrário
    diff = input1 - input2
    dist_sq = torch.sum(torch.pow(diff,2),1)
    dist = torch.sqrt(dist_sq + 1e-10) # Soma 1e-10 pois se o sqrt der 0 o gradiente da NaN
    mdist = margin - dist
    mdist = F.relu(mdist)
    loss = ((1-y) * dist * dist) + (y * mdist * mdist)
    return torch.mean(loss)

def AEContrastive(input, output, embeddings, labels, pick_encoded, pick_labels, margin=1.0, lambda1=1.0):
    mseloss = nn.MSELoss()
    ae_loss = mseloss(input,output)

    A, B, labels_pair = pick_pair(embeddings, labels, pick_encoded, pick_labels)
    contrastive_loss = ContrastiveLoss(A, B, labels_pair, margin=margin)
    return ae_loss + contrastive_loss * lambda1, ae_loss, contrastive_loss

In [7]:
class contrastive_ae(nn.Module):
    def __init__(self, input_neurons, neurons):
        super().__init__()
        self.encoder0 = nn.Linear(input_neurons,64)
        self.encoder1 = nn.Linear(64,32)
        self.encoder2 = nn.Linear(32,16)
        self.encoder3 = nn.Linear(16,neurons)

        self.decoder0 = nn.Linear(neurons,16)
        self.decoder1 = nn.Linear(16,32)
        self.decoder2 = nn.Linear(32,64)
        self.decoder3 = nn.Linear(64,input_neurons)

        self.activation0 = nn.ReLU()
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        encoded = X
        X = self.activation0(self.decoder0(X))
        X = self.activation0(self.decoder1(X))
        X = self.activation0(self.decoder2(X))
        X = self.decoder3(X)


        return X, encoded

In [8]:
class encoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder0 = list(model.children())[0]
        self.encoder1 = list(model.children())[1]
        self.encoder2 = list(model.children())[2]
        self.encoder3 = list(model.children())[3]
        self.activation0 = list(model.children())[8]
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        return X

In [9]:
from torch.utils.data import Dataset, DataLoader,TensorDataset
import random
def build_class_reference_batches(X_train, y_train, batch_size=512):
    classes = torch.unique(y_train).tolist()
    n_classes = len(classes)
    total_samples = len(X_train)
    dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    

    print(classes)

    # Mapeia cada classe para seus índices
    class_to_indices = {
        int(cls.item()): (y_train == cls).nonzero(as_tuple=True)[0].tolist()
        for cls in torch.unique(y_train)
    }

    reference_list = []
    total_batches = len(train_loader)

    for batch_idx in range(total_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_samples)
        batch_indices = set(range(start, end))

        class_samples = []
        for cls in classes:
            candidates = [i for i in class_to_indices[cls] if i not in batch_indices]
            if not candidates:
                raise ValueError(f"Classe {cls} não possui amostras fora do batch {batch_idx}")
            chosen = candidates[torch.randint(len(candidates), (1,)).item()]
            class_samples.append(chosen)

        reference_list.append(class_samples)


    return train_loader, reference_list

In [10]:
def normalize(t):
    t_norm = (t - t.mean()) / t.std()
    return t_norm


In [10]:
array_hidden_classes = [[0],[2],[3],[4],[5]]
filenames = ['0','2','3','4','5']
# array_hidden_classes = [[0]]
# filenames = ['0']
total_classes = len(labels_str)
for i in range(len(array_hidden_classes)):
    filename = f'{filenames[i]}_hidden'
    print(filename)
    hidden_classes = array_hidden_classes[i] #Classes ocultas do treinamento

    hidden_classes_index = list(train[train['Label'].isin(hidden_classes)].index)

    train_hidden = train.loc[hidden_classes_index]

    train_not_hidden = train.drop(hidden_classes_index)

    X_train = train_not_hidden.drop(columns=['Label'])
    y_train = train_not_hidden['Label']
    X_val = val.drop(columns=['Label'])
    y_val = val['Label'].values
    X_test = test.drop(columns=['Label'])
    y_test = test['Label'].values

    torch.manual_seed(123)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)
    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_loader, pick_list = build_class_reference_batches(X_train,y_train, batch_size=512)
    pick_samples = []
    pick_labels = []
    for l in pick_list:
        pick_samples.append(X_train[l])
        pick_labels.append(y_train[l])

    pick_samples = torch.stack(pick_samples)    
    pick_labels = torch.stack(pick_labels)

    print('pick completo')

    neurons = total_classes - len(hidden_classes)
    print(f'{X_train.shape[1]} {neurons}')
    model = contrastive_ae(input_neurons=X_train.shape[1],neurons=neurons)
    model.to(device)

    learning_rate = 0.0001
    opt = optim.Adam(model.parameters(),lr=learning_rate)
    margin = 10 # Parametro de afastamento das classes
    cae_lambda_1 = 0.1 # Parametro de influencia na distancia das classes
    
    
    for epoch in range(250):
        running_loss = 0
        running_ael = 0
        running_cl = 0
        contador = 0
        for data in train_loader:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)
            ps = pick_samples[contador]
            pl = pick_labels[contador]
            ps = ps.to(device)
            pl = pl.to(device)
            contador += 1
            #print(f'{contador}/{len(train_loader)}')
            opt.zero_grad()
            outputs, encoded = model(inputs)
            _, pick_encoded = model(ps)
            tam_pe = pick_encoded.shape[0]
            enc_total = torch.cat((encoded,pick_encoded), dim=0)
            enc_total_norm = enc_total
            encoded_norm = enc_total_norm[:-tam_pe]
            pick_encoded_norm = enc_total_norm[-tam_pe:]
            loss, ael, cl = AEContrastive(input=inputs, output=outputs, embeddings = encoded_norm, labels=labels, pick_encoded = pick_encoded_norm, pick_labels = pl, margin=margin, lambda1=cae_lambda_1)
            #loss = dummyMSELoss(inputs,outputs)
            loss.backward()
            opt.step()
            running_loss += loss.item()
            running_ael += ael.item()
            running_cl += cl.item()
            #print(f'loss = {loss.item()} batch_size = {size}')
        print(f'filename: {filename} Epoch {epoch+1} loss:{running_loss/len(train_loader)} ael:{running_ael/len(train_loader)} cl:{running_cl/len(train_loader)}')

    model_e = encoder(model=model)
    model_e.to(device)

    model_e.eval()

    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_encoded = model_e(X_train.to(device))

    train_encoded = train_encoded.detach().cpu().numpy()

    train_encoded_df = pd.DataFrame(train_encoded)
    train_encoded_df['Label'] = y_train.detach().cpu().numpy().astype(int)
    display(train_encoded_df)

    score = davies_bouldin_score(train_encoded_df.drop(columns=['Label']), train_encoded_df['Label'])
    print("Davies-Bouldin Score:", score)

    X_val = torch.tensor(np.array(X_val), dtype=torch.float)

    val_encoded = model_e(X_val.to(device))

    val_encoded = val_encoded.detach().cpu().numpy()

    val_encoded_df = pd.DataFrame(val_encoded)

    val_encoded_df['Label'] = y_val

    display(val_encoded_df)

    X_test = torch.tensor(np.array(X_test), dtype=torch.float)

    test_encoded = model_e(X_test.to(device))

    test_encoded = test_encoded.detach().cpu().numpy()

    test_encoded_df = pd.DataFrame(test_encoded)

    test_encoded_df['Label'] = y_test

    display(test_encoded_df)

    train_encoded_df.to_csv(f'train_encoded_CADE_cod_joined_attacks_pick_class_squared_normalized_{filename}.csv',index=False)
    val_encoded_df.to_csv(f'val_encoded_CADE_cod_joined_attacks_pick_class_squared_normalized_{filename}.csv',index=False)
    test_encoded_df.to_csv(f'test_encoded_CADE_cod_joined_attacks_pick_class_squared_normalized_{filename}.csv',index=False)

0_hidden
cuda
[1.0, 2.0, 3.0, 4.0, 5.0]
pick completo
82 5
filename: 0_hidden Epoch 1 loss:3.7306455969090333 ael:0.021364391333919784 cl:37.09281147616168
filename: 0_hidden Epoch 2 loss:1.5497389359191882 ael:0.0062059268673812125 cl:15.435329832859784
filename: 0_hidden Epoch 3 loss:0.7593025748306256 ael:0.004646793432628261 cl:7.546557675913558
filename: 0_hidden Epoch 4 loss:0.6299241629342518 ael:0.00453544401115859 cl:6.253887071467387
filename: 0_hidden Epoch 5 loss:0.5573538263394763 ael:0.004298306134031682 cl:5.530555097220455
filename: 0_hidden Epoch 6 loss:0.5067604924767813 ael:0.004174075386082987 cl:5.025864078976481
filename: 0_hidden Epoch 7 loss:0.47315727904637017 ael:0.004199272008804792 cl:4.689579984424482
filename: 0_hidden Epoch 8 loss:0.44938533435089184 ael:0.004228557836402401 cl:4.451567684504663
filename: 0_hidden Epoch 9 loss:0.43195265767690016 ael:0.004207709813554274 cl:4.2774494072625675
filename: 0_hidden Epoch 10 loss:0.42040831469332457 ael:0.0039

C:\Users\linco\AppData\Local\Temp\ipykernel_50728\310156585.py:91: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  X_train = torch.tensor(np.array(X_train), dtype=torch.float)
C:\Users\linco\AppData\Local\Temp\ipykernel_50728\310156585.py:92: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_train = torch.tensor(np.array(y_train), dtype=torch.float)


,0,1,2,3,4,Label
0,2.627134,3.495268,12.167979,-15.949884,4.420605,1
1,1.253443,1.723965,6.202833,-7.987063,2.176383,2
2,2.446810,3.246908,12.590688,-16.675304,4.283470,1
3,-0.087627,0.432573,19.349188,-26.161076,2.376436,4
4,1.253602,1.724164,6.203566,-7.988064,2.176653,2
...,...,...,...,...,...,...
1270932,-0.888324,-0.917520,0.388868,-0.256036,-0.957226,3
1270933,1.253349,1.723847,6.202387,-7.986454,2.176220,2
1270934,1.253399,1.723910,6.202624,-7.986778,2.176306,2
1270935,1.253358,1.723859,6.202437,-7.986522,2.176238,2


Davies-Bouldin Score: 1.158241855420865


,0,1,2,3,4,Label
0,2.412533,3.300767,12.190971,-16.070925,4.314942,0
1,1.254101,1.724788,6.205111,-7.990170,2.177426,2
2,2.500060,3.488555,12.417381,-16.266672,4.352566,1
3,-1.168649,-1.183880,0.005369,0.337964,-1.234653,3
4,2.473984,3.479240,12.343463,-16.244940,4.384403,1
...,...,...,...,...,...,...
519951,2.551496,3.496222,12.542336,-16.484617,4.416090,1
519952,0.230693,0.867971,18.620375,-25.150749,2.811499,4
519953,2.909551,3.804221,11.587043,-15.182983,4.614385,0
519954,2.563285,3.421139,12.210402,-15.991514,4.325774,1


...

C:\Users\linco\AppData\Local\Temp\ipykernel_50728\310156585.py:91: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  X_train = torch.tensor(np.array(X_train), dtype=torch.float)
C:\Users\linco\AppData\Local\Temp\ipykernel_50728\310156585.py:92: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_train = torch.tensor(np.array(y_train), dtype=torch.float)


,0,1,2,3,4,Label
0,-23.714449,8.807453,9.138328,-19.004087,3.780637,1
1,-22.709101,3.508615,3.171620,-17.444405,-2.789791,0
2,-7.825279,4.658035,4.985006,-6.610040,3.668217,2
3,-23.862188,8.838615,9.208631,-19.059681,3.833048,1
4,-15.927458,6.346906,6.601358,-12.859289,3.158356,4
...,...,...,...,...,...,...
2079255,-22.707651,3.479950,3.140583,-17.437237,-2.825277,0
2079256,-7.823956,4.658403,4.985418,-6.609132,3.669116,2
2079257,-7.822762,4.658734,4.985790,-6.608313,3.669927,2
2079258,-22.673147,3.509168,3.172979,-17.418961,-2.777680,0


Davies-Bouldin Score: 0.035277358125125904


,0,1,2,3,4,Label
0,-22.751348,3.504588,3.124305,-17.439400,-2.854688,0
1,-7.829831,4.657374,4.984282,-6.613243,3.665909,2
2,-23.888180,8.823393,9.132940,-19.044310,3.730878,1
3,-1.421076,0.359308,0.295145,-1.316149,0.103529,3
4,-23.708405,8.736197,9.083659,-18.970789,3.756942,1
...,...,...,...,...,...,...
519951,-23.876476,8.832364,9.187076,-19.041460,3.770055,1
519952,-16.094866,6.348246,6.614672,-12.959662,3.054944,4
519953,-22.712402,3.489101,3.150401,-17.442572,-2.815310,0
519954,-23.803226,8.814026,9.114130,-19.050419,3.742764,1


,0,1,2,3,4,Label
0,-22.715939,3.508509,3.171360,-17.449242,-2.792096,0
1,-22.753258,3.512852,3.134067,-17.441826,-2.844079,0
2,-23.617523,8.736773,9.066226,-18.830179,3.722422,1
3,-24.208742,8.956999,9.303881,-19.296675,3.826121,1
4,-22.751255,3.523521,3.176878,-17.450157,-2.851921,0
...,...,...,...,...,...,...
649942,-23.812614,8.770892,9.115119,-19.051184,3.764956,1
649943,-22.757195,3.545540,3.196683,-17.461575,-2.831181,0
649944,-22.748598,3.531909,3.186722,-17.449286,-2.840628,0
649945,-1.502956,0.385110,0.296889,-1.386206,0.078232,3
